# Heat Science Calculation & Validation Notebook

## Goal
Validate the heat-risk calculations used by the project and produce a shared set of test vectors that both the Python implementation and Azam's TypeScript implementation must strictly match.

## Assumptions & Scope
- **Heat Index & Bands:** Strictly following PRD thresholds.
- **Recovery Logic:** Uses persistence filtering with `direction="below"` (project assumption around 26.7°C / 80°F, not a medical/standard consensus).
- **Occupational Guidance / OSHA Triggers:** Marked as `TODO/VERIFY` until final verification per PRD requirements.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths setup for shared test vectors
RESEARCH_DIR = Path("../")
TEST_VECTORS_DIR = RESEARCH_DIR / "test-vectors"
TEST_VECTORS_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = TEST_VECTORS_DIR / "heat_science_test_vectors.csv"

print(f"Setup complete. Target CSV path: {CSV_PATH.resolve()}")

Setup complete. Target CSV path: C:\Users\nasse\temperature-api-quickstart\test-vectors\heat_science_test_vectors.csv


In [2]:
# Initial Test Vector Table (Aligned with PRD Inputs)
# Columns:
# - test_id: Unique identifier
# - heat_index_f: Heat Index in Fahrenheit
# - wet_bulb_c: Wet-Bulb Temperature in Celsius
# - humidity_pct: Relative Humidity (%)
# - solar_irradiance: Solar Irradiance (W/m²)
# - persistence_h: Duration/Persistence threshold in hours
# - expected_category: Expected Heat Risk Band
# - expected_recovery: Boolean indicating if recovery threshold is met

test_vectors_data = [
    {
        "test_id": "TC-001",
        "heat_index_f": 82.0,
        "wet_bulb_c": 22.0,
        "humidity_pct": 50,
        "solar_irradiance": 400,
        "persistence_h": 2,
        "expected_category": "Caution",
        "expected_recovery": False
    },
    {
        "test_id": "TC-002",
        "heat_index_f": 95.0,
        "wet_bulb_c": 26.0,
        "humidity_pct": 65,
        "solar_irradiance": 700,
        "persistence_h": 3,
        "expected_category": "Extreme Caution",
        "expected_recovery": False
    },
    {
        "test_id": "TC-003",
        "heat_index_f": 105.0,
        "wet_bulb_c": 29.0,
        "humidity_pct": 70,
        "solar_irradiance": 850,
        "persistence_h": 4,
        "expected_category": "Danger",
        "expected_recovery": False
    },
    {
        "test_id": "TC-004",
        "heat_index_f": 130.0,
        "wet_bulb_c": 33.0,
        "humidity_pct": 80,
        "solar_irradiance": 950,
        "persistence_h": 5,
        "expected_category": "Extreme Danger",
        "expected_recovery": False
    },
    {
        "test_id": "TC-005",
        "heat_index_f": 78.0,
        "wet_bulb_c": 19.0,
        "humidity_pct": 40,
        "solar_irradiance": 200,
        "persistence_h": 6,
        "expected_category": "Normal",
        "expected_recovery": True
    }
]

df_test_vectors = pd.DataFrame(test_vectors_data)
df_test_vectors.to_csv(CSV_PATH, index=False)
print("Initial test vectors saved successfully.")
df_test_vectors

Initial test vectors saved successfully.


,test_id,heat_index_f,wet_bulb_c,humidity_pct,solar_irradiance,persistence_h,expected_category,expected_recovery
0,TC-001,82.0,22.0,50,400,2,Caution,False
1,TC-002,95.0,26.0,65,700,3,Extreme Caution,False
2,TC-003,105.0,29.0,70,850,4,Danger,False
3,TC-004,130.0,33.0,80,950,5,Extreme Danger,False
4,TC-005,78.0,19.0,40,200,6,Normal,True


In [3]:
def classify_heat_index(heat_index_f: float) -> str:
    """
    Classifies Heat Index (in Fahrenheit) according to NWS standards from PRD Section 7.
    """
    if heat_index_f < 80.0:
        return "Normal"
    elif 80.0 <= heat_index_f < 90.0:
        return "Caution"
    elif 90.0 <= heat_index_f < 103.0:
        return "Extreme Caution"
    elif 103.0 <= heat_index_f < 125.0:
        return "Danger"
    else:  # >= 125.0
        return "Extreme Danger"


def evaluate_overnight_recovery(
    hours_below_threshold: float, 
    recovery_threshold_c: float = 26.7
) -> bool:
    """
    Evaluates whether sufficient overnight thermal recovery occurred.
    
    Note: Threshold of 26.7°C (80°F) is a project assumption derived from heat-health
    research, not a strict medical/agency standard.
    
    Returns True if hours below threshold >= 6 hours.
    """
    MIN_RECOVERY_HOURS = 6.0
    return hours_below_threshold >= MIN_RECOVERY_HOURS


# TODO/VERIFY: Occupational Guidance & OSHA triggers are left out of core classification
# to avoid hard-coding unverified federal rules prior to official publication.

print("Classification and Recovery logic functions defined successfully.")

Classification and Recovery logic functions defined successfully.


In [4]:
# Load test vectors from shared CSV
df_vectors = pd.read_csv(CSV_PATH)

# Run classification & recovery validation
passed_count = 0
total_count = len(df_vectors)

print("--- 🧪 Running Heat Science Validation Suite ---")

for idx, row in df_vectors.iterrows():
    calculated_cat = classify_heat_index(row["heat_index_f"])
    calculated_rec = evaluate_overnight_recovery(row["persistence_h"])
    
    cat_match = calculated_cat == row["expected_category"]
    rec_match = calculated_rec == row["expected_recovery"]
    
    if cat_match and rec_match:
        passed_count += 1
        print(f"✅ {row['test_id']}: PASSED (Category: {calculated_cat}, Recovery: {calculated_rec})")
    else:
        print(f"❌ {row['test_id']}: FAILED")
        if not cat_match:
            print(f"    Expected Cat: {row['expected_category']}, Got: {calculated_cat}")
        if not rec_match:
            print(f"    Expected Recovery: {row['expected_recovery']}, Got: {calculated_rec}")

print("-" * 50)
print(f"Validation Result: PASS {passed_count}/{total_count} test cases")

if passed_count == total_count:
    print(f"🎉 Success! Shared test vectors updated at {CSV_PATH.resolve()}")

--- 🧪 Running Heat Science Validation Suite ---
✅ TC-001: PASSED (Category: Caution, Recovery: False)
✅ TC-002: PASSED (Category: Extreme Caution, Recovery: False)
✅ TC-003: PASSED (Category: Danger, Recovery: False)
✅ TC-004: PASSED (Category: Extreme Danger, Recovery: False)
✅ TC-005: PASSED (Category: Normal, Recovery: True)
--------------------------------------------------
Validation Result: PASS 5/5 test cases
🎉 Success! Shared test vectors updated at C:\Users\nasse\temperature-api-quickstart\test-vectors\heat_science_test_vectors.csv
